## Modelling the Effect of the Compression of Fouling Layer
##### This script builds a mathematical model of the fouling layer compression using filtration cake theory. It estimates model empirical parameters by fitting experimental data to the model equations. The model subsequently predicts the pressure drop increase and flux decline as a function of time.

#### Table of Contents

1. [Estimation of the Model Parameters from Experimental Data](#1-estimation-of-the-model-parameters-from-experimental-data)  
2. [Solving all the Governing Equations in the Mathematical Model](#2-solving-all-the-governing-equations-in-the-mathematical-model)  
3. [Prediction of Flux Decline for a Constant Pressure Run](#3-prediction-of-flux-decline-for-a-constant-pressure-run)  
4. [Prediction of Total Pressure Drop Increase in a Constant Flux Run](#4-prediction-of-total-pressure-drop-increase-in-a-constant-flux-run)  

In [ ]:
# Import necessary python libraries
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from numba import njit, prange

## 1. Estimation of the Model Parameters from Experimental Data

In [ ]:
def estimated_parameters():
    # Input experimental data (these are hypothetical values, replace with actual experimental values if needed)
    Exp_t = np.array([300, 600, 900, 1200, 1500, 1800, 2100, 2400,
                      2700, 3000, 3300, 3600, 3900, 4200, 4500,
                      4800, 5100, 5400])

    Exp_J1 = np.array([5.22222E-05, 3.67778E-05, 3.00741E-05, 2.58333E-05,
                       2.32444E-05, 2.12222E-05, 1.96825E-05, 1.83333E-05, 1.73827E-05, 1.64889E-05,
                       1.56566E-05, 1.4963E-05, 1.44103E-05, 1.38889E-05, 1.34074E-05, 0.000013, 1.25621E-05, 1.22469E-05])

    Exp_DeltaPc = np.array([33481.93333, 35416.98438, 36238.53236, 36781.70458,
                            37091.31275, 37352.03542, 37538.26589, 37713.58443, 37827.31111, 37938.66142,
                            38044.58, 38132.84549, 38201.26429, 38265.72869, 38327.02956, 38375.57557,
                            38432.78838, 38470.06491])

    # Initial guess
    initial_bestx = np.array([1.37, 0.15, 0.3877, 0.055, 1.89E-14, 0.3])
    # Optimization
    result = minimize(lambda bestx: para_function(bestx, Exp_t, Exp_J1,
                                                  Exp_DeltaPc), initial_bestx, method='Nelder-Mead')
    est_params = result.x
    # Display results
    print("Estimated Parameters:")
    print(f"n = {est_params[0]:.4f}")
    print(f"beta = {est_params[1]:.4f}")
    print(f"tau = {est_params[2]:.4f}")
    print(f"theta = {est_params[3]:.4f}")
    print(f"k0 = {est_params[4]:.4e}")
    print(f"eps0 = {est_params[5]:.4f}")
    # Plot results
    plt.figure()
    plt.plot(Exp_t / 60, Exp_J1, 'ro', label='Experimental Data')
    y_fit = model_funct(est_params, Exp_DeltaPc, Exp_t)
    plt.plot(Exp_t / 60, y_fit, 'b-', linewidth=2, label='Model')
    plt.xlabel('Time (minutes)')
    plt.ylabel('Permeate flux (m3/m2.s)')
    plt.title('Data Fitting for J vs t')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Calculate goodness of fit statistics
    y_exp = Exp_J1
    y_model = model_funct(est_params, Exp_DeltaPc, Exp_t)

    # R-squared calculation
    ss_res = np.sum((y_exp - y_model) ** 2)  # SSE
    ss_tot = np.sum((y_exp - np.mean(y_exp)) ** 2)
    r_squared = 1 - (ss_res / ss_tot)

    # Adjusted R-squared
    n_points = len(y_exp)
    n_params = len(est_params)
    adj_r_squared = 1 - (1 - r_squared) * (n_points - 1) / (n_points - n_params - 1)

    # Root Mean Square Error (RMSE)
    rmse = np.sqrt(ss_res / n_points)

    # Mean Absolute Error (MAE)
    mae = np.mean(np.abs(y_exp - y_model))

    print("\nGoodness of Fit Statistics:")
    print(f"R-squared: {r_squared:.6f}")
    print(f"Adjusted R-squared: {adj_r_squared:.6f}")
    print(f"SSE: {ss_res:.6e}")
    print(f"RMSE: {rmse:.6e}")
    print(f"MAE: {mae:.6e}")
    y_exp = Exp_J1
    y_model = model_funct(est_params, Exp_DeltaPc, Exp_t)

def model_funct(bestx, Exp_DeltaPc, Exp_t):
    n, beta, tau, theta, k0, eps0 = bestx
    s = 0.00398
    mu = 0.00089
    DeltaP = 40000
    A = 1 - n - beta
    Pa = (tau * np.exp((DeltaP / 1000) * theta)) * 1000
    C = (2 * mu * s * A) / (k0 * Pa * (eps0 - s))
    J_fitted = -(((1 + (Exp_DeltaPc / Pa)) ** A - 1) /
                 np.sqrt(C * ((1 + (Exp_DeltaPc / Pa)) ** A - 1) * Exp_t))
    return J_fitted

def para_function(bestx, Exp_t, Exp_J1, Exp_DeltaPc):
    J_model = model_funct(bestx, Exp_DeltaPc, Exp_t)
    return np.sum((Exp_J1 - J_model) ** 2)  # Sum of squared errors

if __name__ == "__main__":
    estimated_parameters()

## 2. Solving all the Governing Equations in the Mathematical Model

In [ ]:
# Parameters (replace with actual values from experimental data fitting)
n = 1.8213                             # Example value for n from data fitting
beta = 0.1671                          # Example value for beta from data fitting
tau = 0.3660                           # Example value for tau from data fitting
theta = 0.0264                         # Example value for theta from data fitting
epsilon_so = 0.3010                    # Example value for epsilon_so from data fitting
k_o = 1.8739e-14                       # Example value for k_o from data fitting
DeltaP = 50000                         # Example value for Delta P from data fitting
mu = 0.00089                           # Example value for mu
Rm = 1.37e11                           # Example value for Rm
s = 0.00398                            # Example value for s
rhos = 1095.2                          # Example value for rhos
A = 1 - n - beta
Pa = (tau * np.exp((DeltaP / 1000) * theta)) * 1000
C = (2 * mu * s * A) / (k_o * Pa * (epsilon_so - s))
alpha_o = 1 / (epsilon_so * k_o * rhos)

# Time settings
t_start = 100
t_end = 90 * 60
dt = 100
t_values = np.arange(t_start, t_end + dt, dt)
num_steps = len(t_values)

# Preallocate
J_values = np.zeros(num_steps)
DeltaPc_values = np.zeros(num_steps)

# Solve for J and DeltaPc at each time step
x0 = [100, 5e-5]  # Initial guess

for idx, t in enumerate(t_values):
    def F(x):
        return [
            x[0] - DeltaP + (mu * Rm * x[1]),
            x[1] + (((1 + (x[0] / Pa)) ** A - 1) / np.sqrt(C * ((1 + (x[0] / Pa)) ** A - 1) * t))
        ]
    m, _, flag, _ = fsolve(F, x0, full_output=True)
    DeltaPc_values[idx] = m[0]
    J_values[idx] = m[1]
    x0 = m  # Use previous solution as next initial guess

# f, Ps, alpha, epsilon calculations
f_values = np.arange(0, 1.01, 0.01)
num_f = len(f_values)
Ps_values = np.zeros((num_f, num_steps))
alpha_values = np.zeros((num_f, num_steps))
epsilon_values = np.zeros((num_f, num_steps))

for j in range(num_steps):
    DeltaPc = DeltaPc_values[j]
    V = (1 + DeltaPc / Pa) ** A
    for i, f in enumerate(f_values):
        # Ps
        Ps0 = 40000
        def F1(Ps):
            return (Ps / Pa) - (1 + (1 - f) * (V - 1)) ** (1 / A) + 1
        Ps_sol = fsolve(F1, Ps0)[0]
        Ps_values[i, j] = Ps_sol

        # epsilon
        epsilon0 = 0.5
        def F2(epsilon):
            return epsilon - 1 + epsilon_so * ((V - 1) * (1 - f) + 1) ** (beta / A)
        epsilon_sol = fsolve(F2, epsilon0)[0]
        epsilon_values[i, j] = epsilon_sol

        # alpha
        alpha0 = 4e13
        def F3(alpha):
            return alpha - alpha_o * ((V - 1) * (1 - f) + 1) ** (n / A)
        alpha_sol = fsolve(F3, alpha0)[0]
        alpha_values[i, j] = alpha_sol

# Plot Permeate flux vs Time
plt.figure(1)
plt.plot(t_values / 60, J_values * 1000 * 3600)
plt.xlabel('Time (minutes)')
plt.ylabel('Permeate flux (LMH)')
plt.title('J vs Time')
plt.grid(True)

# Plot solid compressive pressure profile across the fouling layer
plt.figure(2)
for r in range(num_steps):
    plt.plot(f_values, Ps_values[:, r])
plt.xlabel('Relative cake thickness, x/L')
plt.ylabel('Solid compressive pressure (kPa)')
plt.title('Ps vs Time')
plt.grid(True)

# Plot porosity profile across the cake layer
plt.figure(3)
for r in range(num_steps):
    plt.plot(f_values, epsilon_values[:, r])
plt.xlabel('Relative cake thickness, x/L')
plt.ylabel('Porosity')
plt.title('Porosity vs Time')
plt.grid(True)

plt.show()

## 3. Prediction of Flux Decline for a Constant Pressure Run

In [ ]:
# Define the parameters (replace with actual values)
n_val = 1.37       # Example value for n
beta = 0.15        # Example value for beta
tau = 0.2877       # Example value for tau
theta = 0.022      # Example value for theta
epsilon_so = 0.27  # Example value for epsilon_so
k_o = 1.89e-14     # Example value for k_o
DeltaP = 50000     # Example value for Delta P
mu = 0.00089       # Example value for mu
Rm = 1.37e11       # Example value for Rm
s = 0.00398

A = 1 - n_val - beta
Pa = (tau * np.exp((DeltaP / 1000) * theta)) * 1000
C = (2 * mu * s * A) / (k_o * Pa * (epsilon_so - s))

# Define the time interval and step
t_start = 100         # start time
t_end = 90 * 60       # end time in seconds (90 minutes)
dt = 100              # time step in seconds

t_values = np.arange(t_start, t_end + dt, dt)
y = np.zeros((len(t_values), 2))

# Loop through time steps
for i, t in enumerate(t_values):
    # Initial guesses for [DeltaPc, J]
    x0 = [100, 5e-5]
    
    # Define system of equations
    def F(x):
        return [
            x[0] - DeltaP + (mu * Rm * x[1]),
            x[1] + (((1 + (x[0] / Pa))**A - 1) /
                    np.sqrt(C * (((1 + (x[0] / Pa))**A) - 1) * t))
        ]
    
    # Solve using fsolve
    sol = fsolve(F, x0)
    y[i, 0] = sol[0]  # DeltaPc
    y[i, 1] = sol[1]  # J

# Extract results
DeltaPc_values = y[:, 0]
J_values = y[:, 1]

# Plot J vs. t
plt.figure()
plt.plot(t_values / 60, J_values * 1000 * 3600)  # LMH conversion
plt.xlabel('Time (minutes)')
plt.ylabel('Permeate flux (LMH)')
plt.title('Permeate flux vs. Time')
plt.grid(True)
plt.show()

## 4. Prediction of Total Pressure Drop Increase in a Constant Flux Run

In [ ]:
# Define the parameters
n_val = 1.35
beta = 0.15
Pa = 1100
k0 = 1.89e-14
eps0 = 0.3
s = 0.00398
mu = 0.00089
J = 9.722e-6
Rm = 1.12e11

A = 1 - n_val - beta
C = (2 * mu * s * A) / (k0 * Pa * (eps0 - s))

# Define the time interval and step
t_start = 300          # seconds
t_end = 90 * 60        # 90 minutes in seconds
dt = 300               # step size in seconds
t_values = np.arange(t_start, t_end + dt, dt)

# Initialize cumulative P value
P_total = 0
y = np.zeros((len(t_values), 2))  # column 0 = P, column 1 = cumulative P_total

for i, t in enumerate(t_values):
    # Define the function for fsolve
    def F(P):
        return P - ((((J**2) * C * t) + 1)**(1 / A) - 1) * Pa - (mu * Rm * J)

    # Initial guess
    P0 = 100

    # Solve for P
    P_solution = fsolve(F, P0)[0]

    # Accumulate
    P_total += P_solution

    # Store results
    y[i, 0] = P_solution
    y[i, 1] = P_total

# Extract cumulative total pressures
TotalP_values = y[:, 1]

# Plot results
plt.figure()
plt.plot(t_values / 60, TotalP_values / 1000)  # Convert time to minutes, pressure to kPa
plt.xlabel("Time (minutes)")
plt.ylabel("Total pressure drop (kPa)")
plt.title("Total pressure drop vs. Time")
plt.grid(True)
plt.show()